In [1]:
from semanticscholar import SemanticScholar
from crossref.restful import Works
from itertools import product
import pandas as pd
import json
import dblp 
import requests
import time
from requests import HTTPError

In [27]:
# search params
# https://crfm.stanford.edu/helm/lite/latest/#/leaderboard Jan 2, 2024
llm_stanford = [
    "GPT-4",
    "GPT-4-Turbo", 
    "Palmyra-X-V3",
    "Palmyra-X", #base
    "Palmyra", #base
    "PaLM-2 unicorn", #unicorn
    "PaLM 2", #base
    "Palmyra-X-V2",
    "Palmyra X", #base
    "Yi 6B",
    "Yi", #base
    "Mixtral 8x7B",
    "Mixtral", #base
    "Claude v1.3",
    "Claude", #base
    "PaLM-2 bison", #bison
    "Claude 2.0",
    "Llama 2",
    "text-davinci-003",
    "text-davinci", #base
    "Claude 2.1",
    "Claude Instant 1.2",
    "text-davinci-002",
]

# https://huggingface.co/spaces/lmsys/chatbot-arena-leaderboard Jan 2, 2024
llm_arena_elo = [
    "GPT-4-Turbo",
    "GPT-4-0314",
    "GPT-4-0613",
    "Claude 1",
    "Claude 2.0",
    "Mixtral-8x7b-Instruc",
    "Mixtral-8x7b", #base
    "Claude 2.1",
    "GPT-3.5-Turbo-0613",
    "GPT-3.5-Turbo", #base
    "GPT-3.5", #base
    "Gemini Pro",
    "Gemini", #base
    "Claude-Instant-1",
    "Claude-Instant",
    "Tulū-2-DPO-70B",
    "Tulū 2",  #base
    "Yi-34B-Chat",
    "Yi 34B", #base
    "GPT-3.5-Turbo-0314",
    "WizardLM-70B-v1.0",
    "WizardLM",#base
    "Vicuna-33B"
    "Vicuna",
]

llms = llm_stanford + llm_arena_elo
llms = list(set(llms))

primary_keywords = [
    "Generative Artificial Intelligence",
    "Generative AI",
    "GenAI",
    "Gen AI",
    "LLM",
    "LM",
    "large language model",
    "language model",
    "small language model",
    "compact language model",
    "foundation model",
    
] + llms

secondary_keywords = [
    "overreliance",
    "over-reliance",
    "misinformation",
    "accessibility",
    "privacy",
    "environmental",
    "explainability",
    "trustworthy",
    "responsible",
]

third_level_keywords1 = ["mitigation", "ethics", "societal", "social", "ethical"]

third_level_keywords2 = ["education"]


start_year = "2017"
sch_year = start_year + "-"
crossref_year = start_year
years = ["2017", "2018", "2019", "2020", "2021", "2022", "2023"]
dblp_formatted_years = " " + "|".join(f"{year}" for year in years)
wos_years = "2017-01-01+2023-12-31"

all_combinations = list(product(primary_keywords, secondary_keywords, third_level_keywords2))
all_combinations_no_tertiary = list(product(primary_keywords, secondary_keywords))

In [28]:
# Prepare DataFrame to store the results
columns = [
    'PaperTitle',
    'ID',    
    'SearchString',
    'SearchedFrom',
]
search_results_df = pd.DataFrame(columns=columns)

In [29]:
# helper functions
sch = SemanticScholar()
# SemanticScholar fields
sch_fields = [
    "title",
    "externalIds",
    "paperId",
    "url",
    "authors",
]

def sch_get_paper_id(sch_result):
    if sch_result['externalIds'].get('DOI') is None:
        if sch_result['externalIds'].get('ArXiv') is None:
            sch_paper_id = "paperid:" + sch_result.get("paperId")
        else:
            sch_paper_id = "DOI:10.48550/arXiv." + sch_result['externalIds'].get('ArXiv')
    else:
        sch_paper_id = f"DOI:{sch_result['externalIds'].get('DOI')}"
        
    return sch_paper_id


def search_semantic_scholar(bulk: bool=False, search_string: str=None, fields=sch_fields, year: str=None):
    if bulk:
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
    else:
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search"
    
    headers = {"Content-Type": "application/json", "x-api-key": "X48LIBLqr86ouHlnMYd3z052sgEm3Nd2wMORPzu5"}
    sch_search_string = search_string
    if fields == "all":
        params = {"query": sch_search_string, "year": year}
    else:
        params = {"query": sch_search_string, "year": year, "fields": ",".join(fields)}
    response = requests.get(api_url, headers=headers, params=params)
    while response.status_code != 200:
        print(f"Semantic Scholar response status code: {response.status_code}, waiting 30 seconds...")
        time.sleep(30)
        response = requests.get(api_url, headers=headers, params=params)
    response_json = response.json()
    return response_json



In [30]:
import time
sch = SemanticScholar()
# crossref_work = Works()
search_results_df = pd.DataFrame(columns=columns)
try:
    for search_string in all_combinations:
        
        print(f"------------------------------- searching for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ------------------------------")
        
        # Semantic Scholar
        # send http request to api
        sch_search_string = ' + '.join(f'"{term}"' for term in search_string)
        response_json = search_semantic_scholar(bulk=True, search_string=sch_search_string, year=sch_year)
        sch_results = response_json["data"]
        print(f">>> Semantic scholar total: {response_json['total']}")
        # Add results to DataFrame
        sch_count = 0
        if sch_results != 0:
            for result in sch_results:
                sch_count += 1
                paper_id = sch_get_paper_id(result)
                new_paper = {
                    'PaperTitle': result['title'],
                    'ID': paper_id,
                    'SearchString': sch_search_string,
                    'SearchedFrom': 'Semantic Scholar'
                }
                print(f"{sch_count}. sch process paper: ", new_paper)
                search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 
        
        # # # Crossref
        # # TODO: too many result
        # crossref_search_string = ' '.join(search_string)
        # cr_search_results = crossref_work.query(crossref_search_string).filter(from_online_pub_date=crossref_year)
        # print(f"Crossref total: {cr_search_results.count()}")
        # crossref_count = 0
        # for cr_result in cr_search_results:
        #     crossref_count += 1
        #     new_paper = {
        #         'PaperTitle': cr_result.get('title'),
        #         'DOI': cr_result.get('DOI'),
        #         'SearchString': crossref_search_string,
        #         'SearchedFrom': 'Crossref'
        #     }
        #     print(f"{crossref_count}. Crossref process paper: ", new_paper)
        #     search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 

        # # search in dblp
        # dblp_search_string = ' '.join(search_string)
        # dblp_search_results = dblp.search(dblp_search_string + dblp_formatted_years)
        # if dblp_search_results is None:
        #     print(f"\n>>> DBLP total: 0")
        # else:
        #     print(f"\n>>> DBLP total: {len(dblp_search_results)}")
        #     dblp_count = 0
        #     for key, result in dblp_search_results.items():
        #         dblp_count += 1
        #         new_paper = {
        #             'PaperTitle': result.get('title'),
        #             'DOI': result.get('doi'),
        #             'SearchString': dblp_search_string,
        #             'SearchedFrom': 'DBLP'
        #         }
        #         print(f"{dblp_count}. DBLP process paper: ", new_paper)
        #         search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
        print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
    search_results_df.to_csv('data/raw/search-results-sch.csv', index=False)
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
except Exception as e:
    search_results_df.to_csv('data/raw/search-results-sch.csv', index=False)
    print(f"An error occurred: {e.with_traceback()}")
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")


------------------------------- searching for ('Generative Artificial Intelligence', 'overreliance', 'education') (0/486) ------------------------------
>>> Semantic scholar total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative Artificial Intelligence', 'overreliance', 'education') (0/486) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative Artificial Intelligence', 'over-reliance', 'education') (1/486) ------------------------------
>>> Semantic scholar total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative Artificial Intelligence', 'over-reliance', 'education') (1/486) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative Artificial Intelligence', 'misinformation', 'education') (2/486) ------------------------------
>>> Semantic scholar total: 3
1. sch process paper:  {'PaperTitle': 'Can we use ChatGPT for Mental Health and Substance Use Education? Examining Its Quality and Po

In [31]:
# dblp WITH tertiary level keywords:
def get_dblp_search_string(keywords):
    search_string = ""
    for keyword in keywords:
        search_string += f'"{keyword}"$ '
    return search_string


search_results_df = pd.DataFrame(columns=columns)
try:
    for search_string in all_combinations:
        print(
            f"------------------------------- searching for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ------------------------------"
        )
        # search in dblp
        dblp_search_string = get_dblp_search_string(search_string)
        dblp_search_results = dblp.search(dblp_search_string + dblp_formatted_years)
        if dblp_search_results is None:
            print(f">>> DBLP total: 0")
        else:
            print(f">>> DBLP total: {len(dblp_search_results)}")
            dblp_count = 0
            for key, result in dblp_search_results.items():
                dblp_count += 1
                if result.get("info").get("doi") is None:
                    paper_id = "url:" + result.get("info").get("url")
                    if "abs" in paper_id:
                        paper_id = "arXiv:" + paper_id.split("abs-")[1].replace(
                            "-", "."
                        )
                else:
                    paper_id = "DOI:" + result.get("info").get("doi")
                new_paper = {
                    "PaperTitle": result.get("info").get("title"),
                    "ID": paper_id,
                    "SearchString": dblp_search_string,
                    "SearchedFrom": "DBLP",
                }
                print(f"{dblp_count}. DBLP process paper: ", new_paper)
                search_results_df = pd.concat(
                    [search_results_df, pd.DataFrame([new_paper])], ignore_index=True
                )
        time.sleep(10)
        print(
            f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n"
        )
    search_results_df.to_csv("data/raw/search-results-dblp.csv", index=False)
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
except Exception as e:
    search_results_df.to_csv("data/raw/search-results-dblp.csv", index=False)
    print(f"An error occurred: {e.with_traceback()}")
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

------------------------------- searching for ('Generative Artificial Intelligence', 'overreliance', 'education') (0/486) ------------------------------
>>> DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative Artificial Intelligence', 'overreliance', 'education') (0/486) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative Artificial Intelligence', 'over-reliance', 'education') (1/486) ------------------------------
>>> DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative Artificial Intelligence', 'over-reliance', 'education') (1/486) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- searching for ('Generative Artificial Intelligence', 'misinformation', 'education') (2/486) ------------------------------
>>> DBLP total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for ('Generative Artificial Intelligence', 'misinformation', 'education') (2/486) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


----------

In [32]:
# dblp WITHOUT tertiary level keywords:
def get_dblp_search_string(keywords):
    search_string = ""
    for keyword in keywords:
        search_string += f'"{keyword}"$ '
    return search_string

search_results_df = pd.DataFrame(columns=columns)
try:
    for search_string in all_combinations_no_tertiary:

        print(f"------------------------------- searching for {search_string} ({all_combinations_no_tertiary.index(search_string)}/{len(all_combinations_no_tertiary)}) ------------------------------")
        # search in dblp
        dblp_search_string = get_dblp_search_string(search_string)
        dblp_search_results = dblp.search(dblp_search_string + dblp_formatted_years)
        if dblp_search_results is None:
            print(f">>> DBLP total: 0")
        else:
            print(f">>> DBLP total: {len(dblp_search_results)}")
            dblp_count = 0
            for key, result in dblp_search_results.items():
                dblp_count += 1
                if result.get('info').get('doi') is None:
                    paper_id = "URL:" + result.get("info").get("url")
                else:
                    paper_id = "DOI:" + result.get('info').get('doi')
                new_paper = {
                    'PaperTitle': result.get('info').get('title'),
                    'ID': paper_id,
                    'SearchString': dblp_search_string,
                    'SearchedFrom': 'DBLP'
                }
                print(f"{dblp_count}. DBLP process paper: ", new_paper)
                search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
        time.sleep(10)
        print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations_no_tertiary.index(search_string)}/{len(all_combinations_no_tertiary)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
    search_results_df.to_csv('data/raw/search-results-dblp-no-tertiary.csv', index=False)
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
except Exception as e:
    search_results_df.to_csv('data/raw/search-results-dblp-no-tertiary.csv', index=False)
    print(f"An error occurred: {e.with_traceback()}")
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

In [33]:
# from urllib.parse import parse_qs, urlparse
# import time
# # Google scholar 
# from serpapi import GoogleSearch
# 
# 
# 
# params = {
#     "engine": "google_scholar",
#     "q": "",
#     "as_ylo": start_year,
#     "api_key": "c029b25c27cf0b387830ed99a0976387fb3f382c068a2f995b0bb92a55c7bb0f"
# }
# search_results_df = pd.DataFrame(columns=columns)
# 
# 
# 
# def gscholar_get_paper_id(gscholar_result):
#     paper_title = gscholar_result.get("title")
#     have_author = gscholar_result.get("publication_info").get("authors")
#     if have_author:
#         paper_author1 = gscholar_result.get("publication_info").get("authors")[0].get("name")
#         paper_author1_last_name = paper_author1.split(" ")[-1]
#     else:
#         paper_author1_last_name, paper_author1 = "", ""
#     search_string = f"{paper_author1} {paper_title}"
#     try:
#         sch_results = search_semantic_scholar(bulk=False, search_string=search_string, year=sch_year)
#     except Exception as e:
#         print(f"An error occurred: {e}")
#         return None
#     if sch_results.get("total") == 0:
#         return f"url:{gscholar_result.get('link')}"
#     sch_paper = sch_results.get("data")[0]
#     sch_paper_authors = sch_paper.get("authors", [])
#     if sch_paper_authors:
#         first_sch_author_last_name = sch_paper_authors[0].get("name").split(" ")[-1] if sch_paper_authors[0].get("name") else ""
#     else:
#         first_sch_author_last_name = ""
# 
#     # Compare authors if available, otherwise compare titles only
#     if ((not have_author or paper_author1_last_name == first_sch_author_last_name) and 
#             sch_paper["title"].lower() == paper_title.lower()):
#         return sch_get_paper_id(sch_paper)
#     else:
#         return f"url:{gscholar_result.get('link')}"
# 
# 
# 
# try:
#     for search_string in all_combinations:
#         start_num = 0  # Initialize start_num for each search string
#         while True:
#             print(f"------- Fetching results starting from {start_num} for {search_string} -------")
#             gscholar_search_string = '"Generative AI" AND overreliance AND mitigation'
#             params["q"] = gscholar_search_string
#             params["start"] = start_num  # Set start parameter for pagination
# 
#             search = GoogleSearch(params)
#             gscholar_search_results = search.get_dict()
#             if gscholar_search_results.get("error"):
#                 print(f">>> {gscholar_search_results.get('error')}")
#                 break
#             else:
#                 print(f">>> Google Scholar total: {gscholar_search_results.get('search_information').get('total_results')}")
#                 gscholar_count = 0
#                 for gscholar_result in gscholar_search_results.get("organic_results"):
#                     gscholar_count += 1
#                     gscholar_paper_id = gscholar_get_paper_id(gscholar_result)
#                     new_paper = {
#                         'PaperTitle': gscholar_result.get('title'),
#                         'PaperId': gscholar_paper_id,
#                         'SearchString': gscholar_search_string,
#                         'SearchedFrom': 'Google Scholar'
#                     }
#                     print(f"{start_num + gscholar_count}. Google Scholar process paper: ", new_paper)
#                     search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
# 
#                     # Check if we reached 10 results on the current page
#                     if gscholar_count == 10:
#                         break
# 
#                 serpapi_pagination = gscholar_search_results.get("serpapi_pagination")
#                 if serpapi_pagination and "next" in serpapi_pagination:
#                     next_link = serpapi_pagination["next"]
#                     next_start = parse_qs(urlparse(next_link).query).get("start")
#                     if next_start:
#                         start_num = int(next_start[0])
#                     else:
#                         print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
#                         break  # Break if next_start is not available
#                 else:
#                     print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ searching end for {search_string} ({all_combinations.index(search_string)}/{len(all_combinations)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")
#                     break  # Break if there is no 'next' link
# 
# 
#     search_results_df.to_csv('data/raw/search-results-gscholar.csv', index=False)
#     print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
# except Exception as e:
#     search_results_df.to_csv('data/raw/search-results-gscholar.csv', index=False)
#     print(f"An error occurred: {e.with_traceback()}")
#     print("XXXXXXXXXXXXXXXXXXXXXXXXXXXX search ends XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")





In [34]:
# Web of Science
all_search_results_df = pd.DataFrame()

def get_wos_search_string(keywords):
    search_string = ""
    for keyword in keywords:
        search_string += f'"{keyword}" '
    return search_string

def search_web_of_science(wos_search_string, year_start=2017, year_end=2024):
    def fetch_data(start_record):
        url = 'https://api.clarivate.com/api/wos'
        params = {
            'usrQuery': f'(TS=({wos_search_string})) AND PY=({year_start}-{year_end})',
            'count': 100,
            'firstRecord': start_record,
            'databaseId': 'WOS',
            'links': "false",
        }
        headers = {
            'X-ApiKey': '1c19a6c1114c4ee6f84142bba8040e6bbaa9825b',
            "Content-Type": "application/json"
        }

        try:
            response = requests.get(url, headers=headers, params=params)
            response.raise_for_status()  # Raises an HTTPError for certain status codes
            return response.json()
        except HTTPError as http_err:
            if response.status_code == 429:
                print("Rate limit reached, waiting 30s to retry...")
                time.sleep(30)  # Adjust the sleep time as necessary
                return fetch_data(start_record)  # Retry the request
            else:
                print(f"HTTP error occurred: {http_err}")
                return None
        except Exception as err:
            print(f"Other error occurred: {err}")
            return None

    start_record = 1
    total_records = None
    paper_count = 0
    search_results = []

    while True:
        data = fetch_data(start_record)
        if data.get("QueryResult").get("RecordsFound") == 0 or data is None:
            total_records = 0
            break

        if total_records is None:
            total_records = data['QueryResult']['RecordsFound']

        records = data['Data']['Records']['records']['REC']
        for record in records:
            uid = record['UID']
            for identifier in record['dynamic_data']['cluster_related']['identifiers']['identifier']:
                if isinstance(identifier, dict) and identifier['type'] == 'doi':
                    uid = f"DOI:{identifier['value']}"
                    break
            titles = record['static_data']['summary']['titles']['title']
            title = next((t['content'] for t in titles if t['type'] == 'item'), 'No title found')
            search_results.append({
                'PaperTitle': title,
                'ID': uid,
                'SearchString': wos_search_string,
                'SearchedFrom': 'Web of Science'
            })
        paper_count += len(records)
        start_record += len(records)

        if paper_count >= total_records:
            break

    return {
        'total': total_records,
        'data': search_results
    }


try:
    for idx, combination in enumerate(all_combinations):
        # https://webofscience.help.clarivate.com/en-us/Content/search-operators.html
        wos_search_string = get_wos_search_string(combination)
        print(
            f"------------------------------- Searching Web of Science for: {wos_search_string} ({idx + 1}/{len(all_combinations)}) ------------------------------")

        response_json = search_web_of_science(wos_search_string=wos_search_string)
        wos_results = response_json['data']
        total_results = response_json['total']
        print(f">>> Web of Science total: {total_results}")

        # Add results to DataFrame
        if wos_results:
            search_results_df = pd.DataFrame(wos_results)
            all_search_results_df = pd.concat([all_search_results_df, search_results_df], ignore_index=True)
            print(search_results_df)
        time.sleep(0.5)
        print(
            f"^^^^^^^^^^^^^^^^^^^^^^^^^^^ Search end for: {wos_search_string} ({idx + 1}/{len(all_combinations)}) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n")

    # Save the complete DataFrame to a CSV file
    all_search_results_df.to_csv('data/raw/search-results-web-of-science.csv', index=False)
    print(
        "XXXXXXXXXXXXXXXXXXXXXXXXXXXX Search ends. Results saved to 'search-results-web-of-science.csv' XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
except Exception as e:
    print(f"Error occurred: {e.with_traceback()}")
    all_search_results_df.to_csv('data/raw/search-results-web-of-science.csv', index=False)

------------------------------- Searching Web of Science for: "Generative Artificial Intelligence" "overreliance" "education"  (1/486) ------------------------------
>>> Web of Science total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ Search end for: "Generative Artificial Intelligence" "overreliance" "education"  (1/486) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- Searching Web of Science for: "Generative Artificial Intelligence" "over-reliance" "education"  (2/486) ------------------------------
>>> Web of Science total: 0
^^^^^^^^^^^^^^^^^^^^^^^^^^^ Search end for: "Generative Artificial Intelligence" "over-reliance" "education"  (2/486) ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


------------------------------- Searching Web of Science for: "Generative Artificial Intelligence" "misinformation" "education"  (3/486) ------------------------------
>>> Web of Science total: 3
                                          PaperTitle  \
0  Can we use ChatGPT for Mental Health and Subst...  

In [3]:
# remove duplicates internally
sch_search_results_df = pd.read_csv('data/raw/search-results-sch.csv')
dblp_search_results_df = pd.read_csv('data/raw/search-results-dblp.csv')
dblp_search_results_df_no_tertiary = pd.read_csv('data/raw/search-results-dblp-no-tertiary.csv')
wos_search_results_df = pd.read_csv('data/raw/search-results-web-of-science.csv')

# First, remove duplicates based on the 'ID'
sch_search_results_df = sch_search_results_df.drop_duplicates(subset=['ID'])
dblp_search_results_df = dblp_search_results_df.drop_duplicates(subset=['ID'])
dblp_search_results_df_no_tertiary = dblp_search_results_df_no_tertiary.drop_duplicates(subset=['ID'])
wos_search_results_df = wos_search_results_df.drop_duplicates(subset=['ID'])

# Next, remove duplicates based on the 'PaperTitle'
sch_search_results_df = sch_search_results_df.drop_duplicates(subset=['PaperTitle'])
dblp_search_results_df = dblp_search_results_df.drop_duplicates(subset=['PaperTitle'])
dblp_search_results_df_no_tertiary = dblp_search_results_df_no_tertiary.drop_duplicates(subset=['PaperTitle'])
wos_search_results_df = wos_search_results_df.drop_duplicates(subset=['PaperTitle'])


sch_search_results_df.to_csv('data/cleaned/search-results-sch-cleaned.csv', index=False)
dblp_search_results_df.to_csv('data/cleaned/search-results-dblp-cleaned.csv', index=False)
dblp_search_results_df_no_tertiary.to_csv('data/cleaned/search-results-dblp-no-tertiary-cleaned.csv', index=False)
wos_search_results_df.to_csv('data/cleaned/search-results-web-of-science-cleaned.csv', index=False)


In [4]:
sch_search_results_df = pd.read_csv('data/cleaned/search-results-sch-cleaned.csv')
# dblp_search_results_df = pd.read_csv('data/cleaned/search-results-dblp-cleaned.csv')
dblp_search_results_df_no_tertiary = pd.read_csv('data/cleaned/search-results-dblp-no-tertiary-cleaned.csv')
wos_search_results_df = pd.read_csv('data/cleaned/search-results-web-of-science-cleaned.csv')

# Concatenate dataframes
frames = [sch_search_results_df, dblp_search_results_df_no_tertiary, wos_search_results_df]

master_df = pd.concat(frames, axis=0, ignore_index=True)

# Drop duplicates based on ID column to avoid duplicate rows for the same ID
master_df.drop_duplicates(subset='ID', keep='first', inplace=True)

# Save to CSV
master_df.to_csv('data/cleaned/search-results-master.csv', index=False)



# Testing stuff

In [ ]:
search_string = 'Ormco Unveils SymetriTM Clear ceramic twin bracket system'
sch_test = sch.search_paper(search_string)

In [ ]:
crossref_work = Works()
search_string = 'Generative AI Social Impact education'
cr_url = crossref_work.query(search_string).filter(from_online_pub_date=crossref_year, 
                                                              type="journal-article").filter(type="journal-article").url
cr_search_results = crossref_work.query(search_string).filter(from_online_pub_date=crossref_year, 
                                                              type="journal-article dd").filter(type="journal-article").count()

In [ ]:
search_string = 'large language model misinformation' + " 2019|2020|2021|2022|2023"
dblp_test = dblp.search(search_string)
for key, result in dblp_test.items():
                new_paper = {
                    'PaperTitle': result.get('info').get('title'),
                    'DOI': result.get('info').get('doi'),
                    'SearchedFrom': 'DBLP'
                }
                print(f"DBLP process paper: ", new_paper)

In [ ]:
api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
headers = {"Content-Type": "application/json"}
params = {"query": "Generative AI + misinformation + social", "year": sch_year, "fields": ",".join(fields)}
response = requests.get(api_url, headers=headers, params=params)
response_json = response.json()

In [ ]:
from serpapi import GoogleSearch
params = {
    "engine": "google_scholar",
    "q": 'Generative AI overreliance mitigation',
    "as_ylo": start_year,
    "start": 20,
    "api_key": "c029b25c27cf0b387830ed99a0976387fb3f382c068a2f995b0bb92a55c7bb0f"
}
def search_semantic_scholar(bulk: bool=False, search_string: str=None, fields=sch_fields, year: str=None):
    if bulk:
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
    else:
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search"
    
    headers = {"Content-Type": "application/json", "x-api-key": "X48LIBLqr86ouHlnMYd3z052sgEm3Nd2wMORPzu5"}
    sch_search_string = search_string
    if fields == "all":
        params = {"query": sch_search_string, "year": year}
    else:
        params = {"query": sch_search_string, "year": year, "fields": ",".join(fields)}
    response = requests.get(api_url, headers=headers, params=params)
    while response.status_code != 200:
        print(f"Semantic Scholar response status code: {response.status_code}, waiting 30 seconds...")
        time.sleep(30)
        response = requests.get(api_url, headers=headers, params=params)
    response_json = response.json()
    return response_json
    

    
def gscholar_get_paper_id(gscholar_result):
    paper_title = gscholar_result.get("title")
    paper_author1 = gscholar_result.get("publication_info").get("authors")[0].get("name").split(" ")[-1]
    paper_author1_last_name = paper_author1.split(" ")[-1]
    search_string = f"{paper_author1} {paper_title}"
    try:
        sch_results = search_semantic_scholar(bulk=False, search_string=search_string, year=sch_year)
    except Exception as e:
        print(f"An error occurred: {e.with_traceback()}")
        return None
    if sch_results.get("total") == 0:
        return f"url:{gscholar_result.get('link')}"
    sch_paper = sch_results.get("data")[0]
    if (paper_author1_last_name in sch_paper["authors"][0].get("name") and 
            sch_paper["title"].lower() == paper_title.lower()):
        return sch_get_paper_id(sch_paper)
    else:
        return f"url:{gscholar_result.get('link')}"
    
    
search = GoogleSearch(params)
gscholar_search_results = search.get_dict()
paper1 = gscholar_search_results.get("organic_results")[1]
print(gscholar_get_paper_id(paper1))


